# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathibhaShaliniS/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content page (content_id), at one snapshot in time — not a page-day, not a client. Every traffic/engagement column (impressions_90d, clicks_90d, ctr, sessions_90d, etc.) is a trailing 90-day aggregate ending at the export date; there's one number per page, not one per day.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/PrathibhaShaliniS/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# grain check: does content_id uniquely identify a row?
duplicate_rows = df["content_id"].duplicated().sum()
print("Total rows:", len(df))
print("Duplicate content_id rows:", duplicate_rows)
print("Distinct clients:", df["client_id"].nunique())


Total rows: 30000
Duplicate content_id rows: 0
Distinct clients: 32


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature — search_volume, competition, cpc, word_count, char_count, days_with_impressions, content_age_days, days_since_last_update, avg_position, impressions_90d, content_type, main_intent, competition_level, freshness_tier, word_count_tier, position_tier — all knowable before any review decision.

Label / proxy — ctr, used only to build the label needs_ctr_review = ctr < tier median ctr. Never used as a feature itself.

Context — content_id (identity, never a feature), client_id (train/test split grouping only, never a feature).
Excluded, with why:

clicks_90d — algebraically tied to ctr (ctr = clicks_90d / impressions_90d × 100); alongside the label it would let a model reconstruct its own answer.
trend_direction / trend_pct — FlyRank's own decline label, not a raw observation.
provider_used / model_used — marked not-for-modeling in this repo's data dictionary.
engagement_rate, scroll_rate, sessions_90d, ai_traffic_pct — post-click behavior; predicting a pre-click outcome (CTR) from what happens after the click is causally backwards, even where it isn't technical leakage.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_fields = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "content_age_days", "days_since_last_update",
    "avg_position", "impressions_90d", "content_type", "main_intent",
    "competition_level", "freshness_tier", "word_count_tier", "position_tier",
]
label_fields = ["ctr"]
context_fields = ["content_id", "client_id"]
excluded_fields = [
    "clicks_90d", "trend_direction", "trend_pct", "provider_used", "model_used",
    "engagement_rate", "scroll_rate", "sessions_90d", "ai_traffic_pct",
]

all_named = set(feature_fields + label_fields + context_fields + excluded_fields)
print("Fields classified:", len(all_named), "of", df.shape[1], "total columns")
print("Not yet classified:", sorted(set(df.columns) - all_named))


Fields classified: 28 of 44 total columns
Not yet classified: ['age_tier', 'age_tier_order', 'ai_sessions_90d', 'char_count_tier', 'clicks_last_30d', 'clicks_prev_30d', 'days_with_sessions', 'engaged_sessions_90d', 'impression_tier', 'impressions_last_30d', 'impressions_prev_30d', 'pageviews_90d', 'scroll_events_90d', 'sessions_last_30d', 'sessions_prev_30d', 'users_90d']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Counts: confirms the 30,000/32-client scale already checked in section 1 (repeated here per-client, to catch a client with an unexpectedly tiny or huge share of rows). Missing values: the two fields with real gaps — search_volume (8.2% missing overall) and word_count (25.7% missing overall) — aren't randomly missing; they're patterned by content_type. Windows: confirms there's no daily date column anywhere in this file — it really is one 90-day aggregate snapshot per page, not a time series, matching the claim in section 1.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# counts: rows per client (catch outlier clients)
per_client = df.groupby("client_id").size()
print("Rows per client — min/median/max:", per_client.min(), per_client.median(), per_client.max())
print()

# missing values: overall, then broken down by content_type to check the pattern
print("Overall missing search_volume:", df["search_volume"].isna().mean().round(3))
print(df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()).round(3))
print()
print("Overall missing word_count:", df["word_count"].isna().mean().round(3))
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(3))
print()

# windows: confirm there is no daily date column -- this really is one snapshot, not a time series
date_like_cols = [c for c in df.columns if "date" in c.lower() or "day" in c.lower()]
print("Date/day-related columns present:", date_like_cols)


Rows per client — min/median/max: 3 567.0 7008

Overall missing search_volume: 0.082
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64

Overall missing word_count: 0.257
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283
Name: word_count, dtype: float64

Date/day-related columns present: ['days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Single snapshot, not a time series. Every traffic field is one 90-day aggregate ending at export — this data can never show whether a page's CTR problem is new, worsening, or already improving. No day-by-day trend can be recovered from it.
No page younger than 90 days exists in this sample (content_age_days ranges from 90 to 564, minimum exactly 90) — unlike a real production GSC extract, there's no "early history" problem visible here, but that also means this dataset can say nothing about how a brand-new page's metrics behave before its window fills up. That's a limit of this sample, not evidence that the problem doesn't exist in general.
The top_3 position tier is unreliable as position data. 1,205 of its rows have avg_position == 0 ("no data") despite the tier label implying a real top-3 ranking — this is why the modeling work in this project excludes top_3 entirely rather than trusting the label.
Client representation is uneven (234 to 1,892 rows per client, from section 3) — any aggregate finding could be dominated by a handful of high-volume clients unless a split or analysis explicitly accounts for client grouping.
No causal claims possible. Everything here is observational — this data can never show that changing a title or snippet causes a CTR increase, only that certain pages currently sit below their tier's typical CTR.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Pages younger than the 90d window itself (content_age_days < 90):",
      (df["content_age_days"] < 90).sum(), "of", len(df))
print("content_age_days range: min", df["content_age_days"].min(),
      "median", df["content_age_days"].median(),
      "max", df["content_age_days"].max())
print()
print("'top_3' tier rows with no real position data (avg_position == 0):",
      ((df["position_tier"] == "top_3") & (df["avg_position"] == 0)).sum(),
      "of", (df["position_tier"] == "top_3").sum())


Pages younger than the 90d window itself (content_age_days < 90): 0 of 30000
content_age_days range: min 90 median 236.0 max 564

'top_3' tier rows with no real position data (avg_position == 0): 1205 of 2321


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.